In [1]:
# mnist_loader.py
# Модуль для подключения и использования базы данных MNIST.

import pickle
import gzip


def load_data():
    f = gzip.open('mnist.pkl.gz', 'rb')
    training_data, validation_data, test_data = pickle.load(f, encoding='latin1')
    f.close()
    return training_data, validation_data, test_data


def load_data_wrapper():
    tr_d, va_d, te_d = load_data()
    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [vectorized_result(y) for y in tr_d[1]]
    training_data = zip(training_inputs, training_results)
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = zip(validation_inputs, va_d[1])
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = zip(test_inputs, te_d[1])
    return training_data, validation_data, test_data


def vectorized_result(j):
    e = np.zeros((10, 1))
    e[j] = 1.0
    return e


In [2]:
# Модуль создания и обучения нейронной сети для распознавания рукописных цифр
# с использованием метода градиентного спуска.

import random
import numpy as np


def sigmoid(z):
    """Определение сигмоидальной функции активации"""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z):
    """Производная сигмоидальной функции"""
    return sigmoid(z) * (1 - sigmoid(z))


class Network(object):
    def __init__(self, sizes):
        """Конструктор класса. sizes - список размеров слоев нейронной сети"""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Вычисление выходных сигналов сети для входа a"""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data):
        """Стохастический градиентный спуск"""
        test_data = list(test_data)
        n_test = len(test_data)
        training_data = list(training_data)
        n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k + mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            print("Epoch {0}: {1} / {2}".format(j, self.evaluate(test_data), n_test))

    def update_mini_batch(self, mini_batch, eta):
        """Шаг градиентного спуска на мини-батче"""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w - (eta / len(mini_batch)) * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / len(mini_batch)) * nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        """Алгоритм обратного распространения"""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        activation = x
        activations = [x]
        zs = []
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l + 1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l - 1].transpose())
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        """Оценка прогресса в обучении"""
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activations, y):
        """Вычисление частных производных стоимостной функции по активациям выходного слоя"""
        return (output_activations - y)

In [3]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 100, 10])
net.SGD(training_data, 10, 10, 3.0, test_data)

Epoch 0: 5815 / 10000
Epoch 1: 6768 / 10000
Epoch 2: 6825 / 10000
Epoch 3: 6862 / 10000
Epoch 4: 7699 / 10000
Epoch 5: 7746 / 10000
Epoch 6: 7732 / 10000
Epoch 7: 7772 / 10000
Epoch 8: 7814 / 10000
Epoch 9: 7791 / 10000


In [4]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 100, 10])
net.SGD(training_data, 10, 100, 1.0, test_data)

Epoch 0: 2625 / 10000
Epoch 1: 3583 / 10000
Epoch 2: 3960 / 10000
Epoch 3: 4486 / 10000
Epoch 4: 4691 / 10000
Epoch 5: 4755 / 10000
Epoch 6: 4788 / 10000
Epoch 7: 4855 / 10000
Epoch 8: 4938 / 10000
Epoch 9: 5375 / 10000


In [5]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 100, 10])
net.SGD(training_data, 10, 1000, 0.01, test_data)

Epoch 0: 1239 / 10000
Epoch 1: 1267 / 10000
Epoch 2: 1302 / 10000
Epoch 3: 1315 / 10000
Epoch 4: 1310 / 10000
Epoch 5: 1333 / 10000
Epoch 6: 1341 / 10000
Epoch 7: 1351 / 10000
Epoch 8: 1365 / 10000
Epoch 9: 1365 / 10000


In [7]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 128, 64, 10])
net.SGD(training_data, 20, 10, 3.0, test_data)

Epoch 0: 8775 / 10000
Epoch 1: 9329 / 10000
Epoch 2: 9425 / 10000
Epoch 3: 9466 / 10000
Epoch 4: 9509 / 10000
Epoch 5: 9522 / 10000
Epoch 6: 9533 / 10000
Epoch 7: 9527 / 10000
Epoch 8: 9600 / 10000
Epoch 9: 9572 / 10000
Epoch 10: 9617 / 10000
Epoch 11: 9626 / 10000
Epoch 12: 9632 / 10000
Epoch 13: 9644 / 10000
Epoch 14: 9614 / 10000
Epoch 15: 9623 / 10000
Epoch 16: 9654 / 10000
Epoch 17: 9645 / 10000
Epoch 18: 9598 / 10000
Epoch 19: 9643 / 10000


In [8]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 30, 10])
net.SGD(training_data, 30, 10, 3.0, test_data)

Epoch 0: 9047 / 10000
Epoch 1: 9207 / 10000
Epoch 2: 9340 / 10000
Epoch 3: 9379 / 10000
Epoch 4: 9427 / 10000
Epoch 5: 9430 / 10000
Epoch 6: 9427 / 10000
Epoch 7: 9456 / 10000
Epoch 8: 9465 / 10000
Epoch 9: 9472 / 10000
Epoch 10: 9470 / 10000
Epoch 11: 9460 / 10000
Epoch 12: 9505 / 10000
Epoch 13: 9501 / 10000
Epoch 14: 9475 / 10000
Epoch 15: 9515 / 10000
Epoch 16: 9518 / 10000
Epoch 17: 9521 / 10000
Epoch 18: 9510 / 10000
Epoch 19: 9516 / 10000
Epoch 20: 9523 / 10000
Epoch 21: 9506 / 10000
Epoch 22: 9514 / 10000
Epoch 23: 9529 / 10000
Epoch 24: 9503 / 10000
Epoch 25: 9518 / 10000
Epoch 26: 9528 / 10000
Epoch 27: 9536 / 10000
Epoch 28: 9526 / 10000
Epoch 29: 9508 / 10000


In [9]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 10])
net.SGD(training_data, 30, 10, 0.001, test_data)

Epoch 0: 1044 / 10000
Epoch 1: 1078 / 10000
Epoch 2: 1139 / 10000
Epoch 3: 1226 / 10000
Epoch 4: 1304 / 10000
Epoch 5: 1401 / 10000
Epoch 6: 1546 / 10000
Epoch 7: 1713 / 10000
Epoch 8: 1834 / 10000
Epoch 9: 1949 / 10000
Epoch 10: 2057 / 10000
Epoch 11: 2121 / 10000
Epoch 12: 2178 / 10000
Epoch 13: 2241 / 10000
Epoch 14: 2265 / 10000
Epoch 15: 2304 / 10000
Epoch 16: 2335 / 10000
Epoch 17: 2372 / 10000
Epoch 18: 2401 / 10000
Epoch 19: 2429 / 10000
Epoch 20: 2452 / 10000
Epoch 21: 2477 / 10000
Epoch 22: 2504 / 10000
Epoch 23: 2523 / 10000
Epoch 24: 2543 / 10000
Epoch 25: 2574 / 10000
Epoch 26: 2596 / 10000
Epoch 27: 2621 / 10000
Epoch 28: 2628 / 10000
Epoch 29: 2648 / 10000


In [10]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 10])
net.SGD(training_data, 30, 10, 3.0, test_data)

Epoch 0: 6571 / 10000
Epoch 1: 6578 / 10000
Epoch 2: 6657 / 10000
Epoch 3: 6685 / 10000
Epoch 4: 6695 / 10000
Epoch 5: 6710 / 10000
Epoch 6: 7563 / 10000
Epoch 7: 7555 / 10000
Epoch 8: 7545 / 10000
Epoch 9: 7562 / 10000
Epoch 10: 7569 / 10000
Epoch 11: 7558 / 10000
Epoch 12: 7559 / 10000
Epoch 13: 7570 / 10000
Epoch 14: 7552 / 10000
Epoch 15: 7564 / 10000
Epoch 16: 7573 / 10000
Epoch 17: 7590 / 10000
Epoch 18: 7571 / 10000
Epoch 19: 7568 / 10000
Epoch 20: 7577 / 10000
Epoch 21: 7563 / 10000
Epoch 22: 7580 / 10000
Epoch 23: 7578 / 10000
Epoch 24: 7578 / 10000
Epoch 25: 7575 / 10000
Epoch 26: 7566 / 10000
Epoch 27: 7594 / 10000
Epoch 28: 7580 / 10000
Epoch 29: 7591 / 10000


In [11]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 64, 10])
net.SGD(training_data, 20, 10, 3.0, test_data)

Epoch 0: 9080 / 10000
Epoch 1: 9312 / 10000
Epoch 2: 9405 / 10000
Epoch 3: 9454 / 10000
Epoch 4: 9483 / 10000
Epoch 5: 9494 / 10000
Epoch 6: 9525 / 10000
Epoch 7: 9518 / 10000
Epoch 8: 9553 / 10000
Epoch 9: 9550 / 10000
Epoch 10: 9557 / 10000
Epoch 11: 9559 / 10000
Epoch 12: 9589 / 10000
Epoch 13: 9593 / 10000
Epoch 14: 9550 / 10000
Epoch 15: 9593 / 10000
Epoch 16: 9622 / 10000
Epoch 17: 9602 / 10000
Epoch 18: 9592 / 10000
Epoch 19: 9588 / 10000


In [12]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 128, 64, 32, 10])
net.SGD(training_data, 20, 10, 4, test_data)

Epoch 0: 9145 / 10000
Epoch 1: 9314 / 10000
Epoch 2: 9396 / 10000
Epoch 3: 9362 / 10000
Epoch 4: 9438 / 10000
Epoch 5: 9501 / 10000
Epoch 6: 9498 / 10000
Epoch 7: 9527 / 10000
Epoch 8: 9536 / 10000
Epoch 9: 9533 / 10000
Epoch 10: 9578 / 10000
Epoch 11: 9607 / 10000
Epoch 12: 9589 / 10000
Epoch 13: 9596 / 10000
Epoch 14: 9620 / 10000
Epoch 15: 9597 / 10000
Epoch 16: 9628 / 10000
Epoch 17: 9588 / 10000
Epoch 18: 9579 / 10000
Epoch 19: 9601 / 10000


In [13]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 128, 64, 32, 10])
net.SGD(training_data, 30, 10, 2.5, test_data)

Epoch 0: 8955 / 10000
Epoch 1: 9233 / 10000
Epoch 2: 9344 / 10000
Epoch 3: 9417 / 10000
Epoch 4: 9504 / 10000
Epoch 5: 9513 / 10000
Epoch 6: 9513 / 10000
Epoch 7: 9563 / 10000
Epoch 8: 9559 / 10000
Epoch 9: 9583 / 10000
Epoch 10: 9598 / 10000
Epoch 11: 9583 / 10000
Epoch 12: 9603 / 10000
Epoch 13: 9617 / 10000
Epoch 14: 9608 / 10000
Epoch 15: 9624 / 10000
Epoch 16: 9637 / 10000
Epoch 17: 9623 / 10000
Epoch 18: 9642 / 10000
Epoch 19: 9640 / 10000
Epoch 20: 9642 / 10000
Epoch 21: 9635 / 10000
Epoch 22: 9618 / 10000
Epoch 23: 9654 / 10000
Epoch 24: 9660 / 10000
Epoch 25: 9644 / 10000
Epoch 26: 9646 / 10000
Epoch 27: 9633 / 10000
Epoch 28: 9667 / 10000
Epoch 29: 9631 / 10000


In [14]:
training_data, validation_data, test_data = load_data_wrapper()
net = Network([784, 128, 64, 32, 10])
net.SGD(training_data, 40, 10, 3.0, test_data)

Epoch 0: 8900 / 10000
Epoch 1: 9117 / 10000
Epoch 2: 9403 / 10000
Epoch 3: 9430 / 10000
Epoch 4: 9459 / 10000
Epoch 5: 9491 / 10000
Epoch 6: 9491 / 10000
Epoch 7: 9421 / 10000
Epoch 8: 9537 / 10000
Epoch 9: 9562 / 10000
Epoch 10: 9542 / 10000
Epoch 11: 9569 / 10000
Epoch 12: 9603 / 10000
Epoch 13: 9601 / 10000
Epoch 14: 9608 / 10000
Epoch 15: 9551 / 10000
Epoch 16: 9554 / 10000
Epoch 17: 9600 / 10000
Epoch 18: 9655 / 10000
Epoch 19: 9644 / 10000
Epoch 20: 9653 / 10000
Epoch 21: 9638 / 10000
Epoch 22: 9631 / 10000
Epoch 23: 9631 / 10000
Epoch 24: 9625 / 10000
Epoch 25: 9657 / 10000
Epoch 26: 9636 / 10000
Epoch 27: 9623 / 10000
Epoch 28: 9665 / 10000
Epoch 29: 9654 / 10000
Epoch 30: 9669 / 10000
Epoch 31: 9688 / 10000
Epoch 32: 9673 / 10000
Epoch 33: 9672 / 10000
Epoch 34: 9666 / 10000
Epoch 35: 9650 / 10000
Epoch 36: 9661 / 10000
Epoch 37: 9689 / 10000
Epoch 38: 9667 / 10000
Epoch 39: 9665 / 10000
